In [1]:
import sys
import random
from tensorflow.compat.v1.keras import backend as KK
import os
import math

from tensorflow import keras
from keras import layers,Model

import matplotlib.pyplot as plt
import tensorflow as tf
import numpy as np
import imageio
from sklearn.model_selection import train_test_split
import pydot

from keras.models import Sequential
from keras.layers import Flatten, Dense, Activation,Lambda
from IPython.display import clear_output
from tensorflow.keras.callbacks import EarlyStopping
from keras.utils import plot_model
import scipy.io
from sklearn.utils import shuffle
import joblib
from sklearn.metrics import mean_squared_error

In [2]:
## randomly select N_budget trajectories from N_total trajectories (can be done with other strategies) with seed_number
def random_sampling(N_budget,N_total,seed_number):
    np.random.seed(seed_number)
    select_idx = np.random.choice(N_total,N_budget,replace = False)
    return select_idx

In [3]:
def log_min_max_scale(X,select_idx):
    logX = np.log(X+11.0)
    X_train = logX[:,:,select_idx]
    X_max = np.nanmax(X_train)
    X_min = np.nanmin(X_train)
    logX_scaled = (logX-X_min)/(X_max-X_min)-0.5
    return logX_scaled

## Loading and preprocessing

In [4]:
kf_ups_5 = scipy.io.loadmat('../../data/1000kf.mat')['kf_ups_5']
## scaling of input ####################################################
log_kf = np.reshape(np.log10(kf_ups_5[[0,2],:,:]),(-1,kf_ups_5.shape[-1]))
params_kf = (log_kf-np.min(log_kf))/(np.max(log_kf)-np.min(log_kf)) - 0.5

In [5]:
params_res = scipy.io.loadmat('../../data/params')['params'] - 0.5
params = np.concatenate((params_kf,params_res),axis = 0)
print("input parameter shape:", params.shape)

input parameter shape: (14, 1000)


In [6]:
P = scipy.io.loadmat('../../data/QoI3')['dPmax']
print("P shape:", P.shape)

P shape: (200, 1, 1000)


In [7]:
L = scipy.io.loadmat('../../data/QoI3')['L06']
L = L[:,0]
print("L shape:", L.shape)

L shape: (1000,)


In [8]:
select_idx = scipy.io.loadmat('../../data/training_idx')['idx_miniMax']# idx_random; idx_sparse; idx_Maxmin; idx_miniMax
select_idx = select_idx[:,0]
print("select_idx shape:", select_idx.shape)

select_idx shape: (100,)


In [9]:
P_scaled = log_min_max_scale(X = P,select_idx = select_idx)
print("P_scaled shape:", P_scaled.shape)

P_scaled shape: (200, 1, 1000)


In [10]:
def set_train(N_mem,N_rec,N_budget,X,select_idx,L,params):
    #####################################################
    N_par = params.shape[0]
    N_MC = params.shape[1]
    d_of_x = 1 # number of variables
    d_input = N_mem *d_of_x + N_par # number of input nodes
    d_output = d_of_x # number of output nodes
    d_outputRNN = N_rec * d_output # total number of output nodes (including recurrence)

    n_data = N_mem + N_rec
    n_burst = 20 # number of segments select from each trajectory
    choice_of_subsampling = 1 # default setting 0: no subsampling; 1: subsampling; 2: select the first (n_data+n_burst) data

    n_hidden = 3 # number of hidden layers
    n_nodes = 10 # number of nodes per hidden layer
    n_epochs = 20_000 # number of epochs for training
    learning_rate = 1e-4 # learning rate
    N_seeds = 10 # number of seeds for ensemble learning
    batch_size = 256 # batch size
    
    #####################################################
    Model_upperdir = '06tanh_logPmax' +'_{}mem'.format(N_mem) +'_{}rec'.format(N_rec) +'_miniMax' \
                    '_{}budget'.format(N_budget)+'_{}'.format(choice_of_subsampling)
    #####################################################
    ## Trajectory subsampling
    QoI_train = np.empty((n_data,d_of_x,0))
    par_train = np.empty((N_par,0))
    if choice_of_subsampling == 0: #no subsampling
        for i in range(N_budget):
            N_seg = L[select_idx[i]] - n_data + 1
            for j in range(N_seg):
                QoI_train = np.concatenate((QoI_train, X[j:j+n_data,:,select_idx[i]:select_idx[i]+1]),axis = 2)
                par_train = np.concatenate((par_train,params[:,select_idx[i],np.newaxis]),axis = 1)
    elif choice_of_subsampling == 1: #subsampling
        for i in range(N_budget):
            N_seg = L[select_idx[i]] - n_data + 1
            if n_burst > N_seg:
                for j in range(N_seg):
                    QoI_train = np.concatenate((QoI_train, X[j:j+n_data,:,select_idx[i]:select_idx[i]+1]),axis = 2)
                    par_train = np.concatenate((par_train,params[:,select_idx[i],np.newaxis]),axis = 1)
            else:
                subsampling_idx = random_sampling(N_budget = n_burst,N_total = N_seg,seed_number = i)
                for j in range(n_burst):
                    QoI_train = np.concatenate((QoI_train, X[subsampling_idx[j]:subsampling_idx[j]+n_data,
                                                             :,select_idx[i]:select_idx[i]+1]),axis = 2)
                    par_train = np.concatenate((par_train,params[:,select_idx[i],np.newaxis]),axis = 1)                
    elif choice_of_subsampling == 2: ## choose first (n_data+n_burst) data
        for i in range(N_budget):
            N_seg = L[select_idx[i]] - n_data + 1
            if n_burst > N_seg:
                for j in range(N_seg):
                    QoI_train = np.concatenate((QoI_train, X[j:j+n_data,:,select_idx[i]:select_idx[i]+1]),axis = 2)
                    par_train = np.concatenate((par_train,params[:,select_idx[i],np.newaxis]),axis = 1)
            else:
                subsampling_idx = np.arange(n_burst)
                for j in range(n_burst):
                    QoI_train = np.concatenate((QoI_train, X[subsampling_idx[j]:subsampling_idx[j]+n_data,
                                                             :,select_idx[i]:select_idx[i]+1]),axis = 2)
                    par_train = np.concatenate((par_train,params[:,select_idx[i],np.newaxis]),axis = 1) 
    else:
        print("specify if subsampling")
    if choice_of_subsampling == 2:  
        print("use the first ", n_data+n_burst, "data")

    print("QoI shape:", QoI_train.shape) ## shape should be (n_data, 1, n_burst * N_budget)
    print("parameter shape:", par_train.shape) ## shape should be (N_par, n_burst * N_budget)
    #####################################################
    ## Input Output
    y_train = np.reshape(QoI_train[-N_rec:,:,:],(-1,QoI_train.shape[2]))
    y_train = y_train.T
    print("y_train shape:", y_train.shape)
    x_train = np.concatenate((np.reshape(QoI_train[:N_mem,:,:],(-1,QoI_train.shape[2])),par_train),axis = 0)
    x_train = x_train.T
    print("x_train shape:", x_train.shape)
    #####################################################
    ## training
    for i in range(N_seeds):
        display(i)

        np.random.seed(i)
        random.seed(i)
        tf.random.set_seed(i)
        Model_dir = Model_upperdir + '/seed_{}/'.format(i)

        session_conf = tf.compat.v1.ConfigProto(intra_op_parallelism_threads=1,
                                      inter_op_parallelism_threads=1)
        sess = tf.compat.v1.Session(graph=tf.compat.v1.get_default_graph(), config=session_conf)
        KK.set_session(sess)
        ## Define Neural Network model----------------------begins------------------- ##
        dense_layer_list=[]
        for __ in range(n_hidden):
            dense_layer = tf.keras.layers.Dense(n_nodes, activation='tanh')
            dense_layer_list += [dense_layer]
        out_layer = tf.keras.layers.Dense(d_output)
        slice_layer_last = tf.keras.layers.Lambda(lambda x: x[:, -d_of_x-N_par:-N_par]) # get the last step state (for ResNet)
        slice_layer_mem = tf.keras.layers.Lambda(lambda x: x[:, d_of_x:-N_par]) # cuts off most recent memory period
        slice_layer_par = tf.keras.layers.Lambda(lambda x: x[:, -N_par:]) # cuts off the parameter part

        inputs = tf.keras.layers.Input(shape=(d_input,))
        x = inputs
        x_last = slice_layer_last(x)
        x_mem = slice_layer_mem(x)
        x_par = slice_layer_par(x)
        for k in range(n_hidden):
            x = dense_layer_list[k](x)
        x = out_layer(x)
        x_add_res = tf.keras.layers.Add()([x,x_last])
        z = x_add_res

        for __ in range(N_rec-1):
            x = tf.keras.layers.Concatenate()([x_mem,x_add_res,x_par])
            x_last = slice_layer_last(x)
            x_mem = slice_layer_mem(x)
            x_par = slice_layer_par(x)
            for k in range(n_hidden):
                x = dense_layer_list[k](x)
            x = out_layer(x)
            x_add_res = tf.keras.layers.Add()([x,x_last])
            z = tf.keras.layers.Concatenate()([z,x_add_res])
        outputs = z
        ## Define Neural Network model----------------------ends--------------------- ##
        model = tf.keras.models.Model(inputs=inputs, outputs=outputs)
        model.compile(optimizer=tf.keras.optimizers.Adam(lr=learning_rate), loss='mean_squared_error')   
        my_callback = tf.keras.callbacks.ModelCheckpoint(Model_dir,monitor="loss",save_best_only=True,save_weights_only=True)
        history = model.fit(x_train, y_train, epochs=n_epochs, batch_size=batch_size, callbacks=my_callback)
        model.load_weights(Model_dir)
        model.save(Model_dir)
        model.summary()

        plt.semilogy(history.history['loss'])
        plt.title('model loss')
        plt.ylabel('loss')
        plt.xlabel('epoch')
        plt.legend(['train'], loc='upper left')
        plt.savefig(Model_dir+'loss_semilog.png', bbox_inches='tight')
        plt.show()
    return

## Training

In [13]:
_ = set_train(N_mem = 20,N_rec = 10,N_budget= 100,X = P_scaled,select_idx = select_idx,L = L,params = params)